# Profile Agent
## AI Financial Advisor Pipeline — Agent 1 of 5

The Profile Agent performs structured intake for each client, combining:
- **Financial capital**: current holdings, savings, investments
- **Human capital**: present value of future earnings based on career type

The discount rate is pulled live from FRED (10-year Treasury yield) rather than hardcoded.

Output is a standardised JSON profile passed downstream to the Research, Allocation, Risk, and Compliance agents.

## 1. Install & Import

In [1]:
!pip install anthropic requests pandas numpy openai pydantic scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 14.4 MB/s eta 0:00:00


In [2]:
import json
import anthropic
import requests
import pandas as pd
import numpy as np
from scipy import stats
from pydantic import BaseModel, field_validator
from typing import Literal
from google.colab import userdata

## 2. Fetch Discount Rate from FRED

The discount rate for human capital valuation is pulled live from FRED using the 10-year Treasury yield (series `DGS10`).
Falls back to 4% if the API call fails.

> **Source:** Board of Governors of the Federal Reserve System (US), Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity [DGS10], retrieved from FRED, Federal Reserve Bank of St. Louis; https://fred.stlouisfed.org/series/DGS10

In [3]:
try:
    FRED_API_KEY = userdata.get('FRED_API')
except Exception:
    FRED_API_KEY = None

In [4]:
def get_discount_rate_from_fred(api_key, fallback=0.04):
    try:
        url = (
            "https://api.stlouisfed.org/fred/series/observations"
            f"?series_id=DGS10&api_key={api_key}"
            "&sort_order=desc&limit=1&file_type=json"
        )
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        value = resp.json()['observations'][0]['value']
        rate = float(value) / 100
        print(f"FRED DGS10 (10Y Treasury): {rate:.4f}")
        return rate
    except Exception as e:
        print(f"FRED fetch failed ({e}), using fallback rate: {fallback}")
        return fallback

DISCOUNT_RATE = get_discount_rate_from_fred(FRED_API_KEY)

FRED DGS10 (10Y Treasury): 0.0449


## 3. Pydantic Data Models

Per the June 11 meeting, all agents must adopt Pydantic `BaseModel` for data validation.
These models enforce schema correctness before any profile data is passed downstream.

- `Ticker` — asset symbol and category
- `Position` — ticker, size, entry price
- `Portfolio` — positions, user, total value property
- `Trade` — ticker, size, price, direction, execution timestamp (timestamp belongs to Trade, not Position)

In [5]:
from datetime import datetime

class Ticker(BaseModel):
    symbol: str
    category: Literal["equity", "bond", "cash", "alternative", "rsu"]

class Position(BaseModel):
    ticker: Ticker
    size: float
    entry_price: float

    @property
    def market_value(self) -> float:
        return round(self.size * self.entry_price, 2)

class Portfolio(BaseModel):
    user: str
    positions: list[Position]

    @property
    def total_value(self) -> float:
        return round(sum(p.market_value for p in self.positions), 2)

class Trade(BaseModel):
    ticker: Ticker
    size: float
    price: float
    is_buy: bool
    execution_time: datetime  # timestamp on Trade, not Position — per June 11 decision

# Quick demo
demo_portfolio = Portfolio(
    user="persona_tech_executive",
    positions=[
        Position(ticker=Ticker(symbol="AAPL", category="equity"), size=100, entry_price=180.0),
        Position(ticker=Ticker(symbol="AGG",  category="bond"),   size=200, entry_price=95.0),
    ]
)
print(f"Demo portfolio total value: ${demo_portfolio.total_value:,.2f}")

Demo portfolio total value: $37,000.00


## 4. Define the 3 Core Baseline Personas

Hardcoded based on the project brief. `income_volatility_sigma` is a quantitative measure of earnings uncertainty — used downstream by the Allocation and Risk agents.

$$\text{Human Capital} = \text{Annual Salary} \times \left( \frac{1 - (1 + r)^{-t}}{r} \right)$$

$$\text{Human Capital} = \text{Annual Salary} \times \left( \frac{1 - (1 + \text{discount_rate})^{-\text{years_to_retirement}}}{\text{discount_rate}} \right)$$

Salary and financial capital figures are grounded in real-world data sources:
- Academic salaries: [AAUP Faculty Compensation Survey 2024](https://www.aaup.org/our-work/research/FCS)
- Tech executive compensation: [Aon Radford Global Technology Survey](https://radford.aon.com/products/surveys/technology-compensation-survey) / [SEC EDGAR Proxy Filings (DEF 14A)](https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&type=DEF+14A)
- Finance professional salaries: [BLS Occupational Employment & Wage Statistics](https://www.bls.gov/oes/current/oes_nat.htm) / [SIFMA Compensation Report](https://www.sifma.org/resources/research/)

In [6]:
# Quantitative income volatility mapping (annualised σ of earnings)
INCOME_VOLATILITY_SIGMA = {
    "High":   0.05,   # tenured/government — very stable
    "Medium": 0.20,   # bonus-driven, market-correlated
    "Low":    0.40,   # RSU/layoff risk — highly variable
}

def compute_human_capital(annual_salary, years_to_retirement, discount_rate=DISCOUNT_RATE):
    if years_to_retirement <= 0:
        return 0.0
    pv = annual_salary * (1 - (1 + discount_rate) ** (-years_to_retirement)) / discount_rate
    return round(pv, 2)

# 3 baseline personas
# Salaries grounded in: AAUP 2024, BLS OES, SIFMA Compensation Report
raw_personas = [
    {
        "client_id": "persona_biology_professor",
        "name": "Biology Professor",
        "age": 45,
        "annual_salary": 95000,       # AAUP 2024: associate prof median $95k
        "years_to_retirement": 20,
        "career_type": "Academia",
        "income_stability": "High",
        "industry_exposure_sector": "Education / Life Sciences",
        "financial_capital": 350000,
        "current_holdings": {"US_bonds": 0.4, "broad_equity": 0.5, "cash": 0.1},
        "investment_horizon_years": 20,
        "risk_tolerance": "Moderate",
        "liquidity_needs": "Low",
        "investment_objective": "Growth",
        "RSU_concentration": 0.0
    },
    {
        "client_id": "persona_tech_executive",
        "name": "Tech Executive",
        "age": 38,
        "annual_salary": 400000,      # Radford 2024: VP-level TC median $380-420k
        "years_to_retirement": 27,
        "career_type": "Technology",
        "income_stability": "Low",
        "industry_exposure_sector": "Technology",
        "financial_capital": 2000000,
        "current_holdings": {"employer_RSU": 0.6, "broad_equity": 0.3, "cash": 0.1},
        "investment_horizon_years": 27,
        "risk_tolerance": "Aggressive",
        "liquidity_needs": "Medium",
        "investment_objective": "Growth",
        "RSU_concentration": 0.6
    },
    {
        "client_id": "persona_financial_professional",
        "name": "Financial Professional",
        "age": 35,
        "annual_salary": 250000,      # SIFMA 2024: mid-career buy-side median $220-270k
        "years_to_retirement": 30,
        "career_type": "Finance",
        "income_stability": "Medium",
        "industry_exposure_sector": "Financial Services",
        "financial_capital": 800000,
        "current_holdings": {"equities": 0.5, "alternatives": 0.3, "bonds": 0.2},
        "investment_horizon_years": 30,
        "risk_tolerance": "Aggressive",
        "liquidity_needs": "Low",
        "investment_objective": "Growth",
        "RSU_concentration": 0.0
    }
]

## 5. Compute Human Capital & Build Profiles

Each profile includes:
- `human_capital_valuation`: PV of future earnings using live FRED discount rate
- `income_volatility_sigma`: quantitative earnings uncertainty score
- `human_capital_type`: classifies income stream as bond-like / mixed / equity-like
- `effective_risk_budget`: portfolio risk capacity accounting for HC type and size
- `liquidity_needs`: minimum liquidity requirement for the Compliance Agent
- `investment_objective`: return target anchor (Growth / Income / Preservation)
- `RSU_concentration`: employer stock concentration for Risk and Compliance agents

In [7]:
# Human capital type mapping based on income volatility sigma
HUMAN_CAPITAL_TYPE = {
    "High":   "bond-like",   # stable, low-volatility income stream
    "Medium": "mixed",       # partially market-correlated
    "Low":    "equity-like"  # growth-oriented, high-volatility income stream
}

# Explicit output schema for downstream agents
PROFILE_SCHEMA = {
    "client_id": "str",
    "career_type": "str",
    "age": "int",
    "financial_capital": "float",
    "human_capital_valuation": "float",
    "total_wealth": "float",
    "human_capital_pct_of_total": "float",
    "income_volatility_sigma": "float",
    "human_capital_type": "str",
    "income_equity_beta": "float",                # β — renamed from human_capital_beta
    "income_equity_correlation": "float | None",  # ρ — Corr(income, S&P 500); pending confirmation
    "implicit_equity_exposure": "float",
    "effective_risk_budget": "float",
    "portfolio_equity_target": "float",           # effective_risk_budget − implicit_equity_exposure
    "industry_exposure_sector": "str",
    "income_stability": "str",
    "current_holdings": "dict",
    "investment_horizon_years": "int",
    "risk_tolerance_level": "str",
    "liquidity_needs": "str",
    "investment_objective": "str",
    "RSU_concentration": "float"
}

def build_profile(persona, beta=None, correlation=None):
    hc = compute_human_capital(persona['annual_salary'], persona['years_to_retirement'])
    total_wealth = persona['financial_capital'] + hc
    sigma = INCOME_VOLATILITY_SIGMA[persona['income_stability']]
    hc_type = HUMAN_CAPITAL_TYPE[persona['income_stability']]

    # Fraction of total wealth that behaves like equity purely from the income side
    implicit_equity_exposure = round(hc * sigma / total_wealth, 3)

    # Higher HC % of total → more bond-like wealth → more equity risk capacity in portfolio
    effective_risk_budget = round((persona['financial_capital'] + hc * (1 - sigma)) / total_wealth, 3)

    # Starting equity target for Allocation Agent — after subtracting implicit HC equity exposure
    portfolio_equity_target = round(effective_risk_budget - implicit_equity_exposure, 3)

    return {
        "client_id":                persona["client_id"],
        "career_type":              persona["career_type"],
        "age":                      persona["age"],
        "financial_capital":        persona["financial_capital"],
        "human_capital_valuation":  hc,
        "total_wealth":             total_wealth,
        "human_capital_pct_of_total": round(hc / total_wealth * 100, 1),
        "income_volatility_sigma":  sigma,
        "human_capital_type":       hc_type,
        "income_equity_beta":       beta,         # β — renamed from human_capital_beta
        "income_equity_correlation": correlation, # ρ — None until confirmed with Niha
        "implicit_equity_exposure": implicit_equity_exposure,
        "effective_risk_budget":    effective_risk_budget,
        "portfolio_equity_target":  portfolio_equity_target,
        "industry_exposure_sector": persona["industry_exposure_sector"],
        "income_stability":         persona["income_stability"],
        "current_holdings":         persona["current_holdings"],
        "investment_horizon_years": persona["investment_horizon_years"],
        "risk_tolerance_level":     persona["risk_tolerance"],
        "liquidity_needs":          persona["liquidity_needs"],
        "investment_objective":     persona["investment_objective"],
        "RSU_concentration":        persona["RSU_concentration"]
    }

profiles = [build_profile(p) for p in raw_personas]

for p in profiles:
    print(json.dumps(p, indent=2))
    print()

{
  "client_id": "persona_biology_professor",
  "career_type": "Academia",
  "age": 45,
  "financial_capital": 350000,
  "human_capital_valuation": 1236825.45,
  "total_wealth": 1586825.45,
  "human_capital_pct_of_total": 77.9,
  "income_volatility_sigma": 0.05,
  "human_capital_type": "bond-like",
  "income_equity_beta": null,
  "income_equity_correlation": null,
  "implicit_equity_exposure": 0.039,
  "effective_risk_budget": 0.961,
  "portfolio_equity_target": 0.922,
  "industry_exposure_sector": "Education / Life Sciences",
  "income_stability": "High",
  "current_holdings": {
    "US_bonds": 0.4,
    "broad_equity": 0.5,
    "cash": 0.1
  },
  "investment_horizon_years": 20,
  "risk_tolerance_level": "Moderate",
  "liquidity_needs": "Low",
  "investment_objective": "Growth",
  "RSU_concentration": 0.0
}

{
  "client_id": "persona_tech_executive",
  "career_type": "Technology",
  "age": 38,
  "financial_capital": 2000000,
  "human_capital_valuation": 6187263.52,
  "total_wealth": 81

## 6. Sanity Check Layer

Guards against unrealistic human capital valuations.
Flags any profile where computed HC deviates more than 50% from the expected actuarial formula.

In [8]:
def sanity_check(profile, raw_persona):
    expected_hc = compute_human_capital(
        raw_persona['annual_salary'],
        raw_persona['years_to_retirement']
    )
    actual_hc = profile['human_capital_valuation']
    deviation = abs(actual_hc - expected_hc) / expected_hc if expected_hc > 0 else 0

    if deviation > 0.5:
        print(f"FLAG: {profile['client_id']} — HC deviation {deviation:.1%} exceeds ±50% threshold")
        print(f"   Expected: ${expected_hc:,.0f} | Got: ${actual_hc:,.0f}")
    else:
        print(f"OK: {profile['client_id']} — HC within acceptable range (deviation: {deviation:.1%})")

for profile, raw in zip(profiles, raw_personas):
    sanity_check(profile, raw)

OK: persona_biology_professor — HC within acceptable range (deviation: 0.0%)
OK: persona_tech_executive — HC within acceptable range (deviation: 0.0%)
OK: persona_financial_professional — HC within acceptable range (deviation: 0.0%)


## 7. Generate Synthetic Personas via Claude, Evaluate via GPT-4o

Synthetic personas are generated by Claude then independently reviewed by GPT-4o to catch:
- Unrealistic salary / wealth combinations
- Holdings inconsistent with stated risk tolerance
- Income stability labels mismatched to career type

This two-model approach avoids circular validation and addresses the hallucination detection
requirement from the project brief.

Salary ranges for synthetic personas are benchmarked against:
- [BLS Occupational Employment & Wage Statistics](https://www.bls.gov/oes/current/oes_nat.htm)
- [BLS National Compensation Survey](https://www.bls.gov/ncs/)

**Data source:** LLM-generated (Claude claude-opus-4-5 + GPT-4o reviewer). Logged here per June 4 action item.

In [9]:
import openai

ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY').strip()
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY').strip()

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)

PERSONA_GENERATION_PROMPT = """
Generate 12 diverse financial planning client personas as a JSON array.
Each persona must have exactly these fields:
- client_id (str, format: persona_<snake_case_name>)
- name (str, human-readable label)
- age (int, between 25 and 60)
- annual_salary (int) — must reflect realistic 2024 US market rates for the career type and age
- years_to_retirement (int, = 65 - age)
- career_type (str)
- income_stability (str, must be exactly one of: "High", "Medium", "Low")
- industry_exposure_sector (str)
- financial_capital (int) — must be plausible given age and career trajectory
- current_holdings (dict with asset class keys summing to 1.0)
- investment_horizon_years (int, same as years_to_retirement)
- risk_tolerance (str, one of: "Conservative", "Moderate", "Aggressive")
- liquidity_needs (str, one of: "Low", "Medium", "High")
- investment_objective (str, one of: "Growth", "Income", "Preservation")
- RSU_concentration (float, 0.0 if no RSUs, otherwise proportion of holdings in employer stock)

Requirements:
- No duplicates of the 3 baseline personas (biology professor, tech executive, financial professional)
- Diverse careers: include healthcare, law, real estate, military, government, arts, engineering, etc.
- Mix of ages, income levels, and risk tolerances
- current_holdings weights must sum to exactly 1.0
- RSU_concentration must match the employer stock proportion in current_holdings (0.0 if none)
- Return only the raw JSON array, no commentary
"""

# Step 1: Claude generates personas
claude_message = anthropic_client.messages.create(
    model="claude-opus-4-5",
    max_tokens=4096,
    messages=[{"role": "user", "content": PERSONA_GENERATION_PROMPT}]
)

raw_text = claude_message.content[0].text.strip()

# Strip markdown code fences if present
if raw_text.startswith("```"):
    raw_text = raw_text.split("```")[1]
    if raw_text.startswith("json"):
        raw_text = raw_text[4:]
    raw_text = raw_text.strip()

print(raw_text[:200])
synthetic_raw_personas = json.loads(raw_text)
print(f"\nClaude generated {len(synthetic_raw_personas)} synthetic personas\n")

# Step 2: GPT-4o reviews each persona independently
REVIEW_PROMPT_TEMPLATE = """
You are an independent financial data auditor. Review this client persona for realism and consistency.

Persona:
{persona}

Check for:
1. Is the salary realistic for this career type and age based on 2024 US market rates?
2. Do the current_holdings match the stated risk tolerance?
3. Is the income_stability label consistent with the career type?
4. Is the financial_capital realistic relative to age and salary?

Respond in this exact JSON format:
{{
  "client_id": "<id>",
  "verdict": "PASS" or "FLAG",
  "issues": ["issue 1", "issue 2"] or []
}}
Return only the raw JSON, no commentary.
"""

review_results = []
for persona in synthetic_raw_personas:
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{
            "role": "user",
            "content": REVIEW_PROMPT_TEMPLATE.format(persona=json.dumps(persona, indent=2))
        }],
        max_tokens=512
    )
    raw_review = response.choices[0].message.content.strip()
    if raw_review.startswith("```"):
        raw_review = raw_review.split("```")[1]
        if raw_review.startswith("json"):
            raw_review = raw_review[4:]
        raw_review = raw_review.strip()

    review = json.loads(raw_review)
    review_results.append(review)
    status = review['verdict']
    issues = review['issues']
    print(f"{status}: {review['client_id']}")
    if issues:
        for issue in issues:
            print(f"     ↳ {issue}")

[
  {
    "client_id": "persona_emergency_room_physician",
    "name": "Emergency Room Physician",
    "age": 42,
    "annual_salary": 350000,
    "years_to_retirement": 23,
    "career_type": "Health

Claude generated 12 synthetic personas

PASS: persona_emergency_room_physician
PASS: persona_corporate_litigation_attorney
PASS: persona_commercial_real_estate_broker
PASS: persona_army_colonel
PASS: persona_federal_policy_analyst
FLAG: persona_freelance_graphic_designer
     ↳ Salary may be slightly above average for a freelance graphic designer at age 29 depending on location and clientele.
     ↳ Income stability 'Low' is consistent with freelance nature of career, however, if salary is stable, there is a potential contradiction.
     ↳ High liquidity needs may not align with aggressive risk tolerance considering cash allocation in current holdings.
     ↳ Financial capital might be low relative to age and supposed salary since savings should typically be higher with no significant RS

## 8. Build Profiles for All Personas & Run Sanity Checks

Builds the standardised profile JSON for all 15 personas (3 baseline + 12 synthetic).
Sanity check flags any persona where HC deviates more than ±50% from the actuarial formula.

Note: `income_equity_beta` and `income_equity_correlation` are both `None` at this stage —
they will be populated in Section 9 after CRSP market return data is loaded and OLS regression is run.

In [10]:
all_raw_personas = raw_personas + synthetic_raw_personas

all_profiles = []
for p in all_raw_personas:
    # Validate income_stability before building
    if p['income_stability'] not in INCOME_VOLATILITY_SIGMA:
        print(f"SKIP: {p['client_id']} — invalid income_stability value: {p['income_stability']}")
        continue

    # Validate new required fields exist
    for field in ['liquidity_needs', 'investment_objective', 'RSU_concentration']:
        if field not in p:
            print(f"WARNING: {p['client_id']} missing field: {field}")

    # Validate current_holdings sum to 1.0
    holdings_sum = round(sum(p['current_holdings'].values()), 2)
    if holdings_sum != 1.0:
        print(f"WARNING: {p['client_id']} holdings sum to {holdings_sum}, not 1.0")

    all_profiles.append(build_profile(p))

print(f"Built {len(all_profiles)} profiles\n")

print("=== Sanity Checks ===")
for profile, raw in zip(all_profiles, all_raw_personas):
    sanity_check(profile, raw)

Built 15 profiles

=== Sanity Checks ===
OK: persona_biology_professor — HC within acceptable range (deviation: 0.0%)
OK: persona_tech_executive — HC within acceptable range (deviation: 0.0%)
OK: persona_financial_professional — HC within acceptable range (deviation: 0.0%)
OK: persona_emergency_room_physician — HC within acceptable range (deviation: 0.0%)
OK: persona_corporate_litigation_attorney — HC within acceptable range (deviation: 0.0%)
OK: persona_commercial_real_estate_broker — HC within acceptable range (deviation: 0.0%)
OK: persona_army_colonel — HC within acceptable range (deviation: 0.0%)
OK: persona_federal_policy_analyst — HC within acceptable range (deviation: 0.0%)
OK: persona_freelance_graphic_designer — HC within acceptable range (deviation: 0.0%)
OK: persona_aerospace_engineer — HC within acceptable range (deviation: 0.0%)
OK: persona_nurse_practitioner — HC within acceptable range (deviation: 0.0%)
OK: persona_restaurant_owner — HC within acceptable range (deviation

## 9. Human Capital Beta & Correlation Estimation

Each persona's human capital is treated as an implicit asset in their total wealth portfolio.
We estimate how sensitive that income stream is to **their specific sector's equity returns** —
i.e., the income equity beta (β) and income equity correlation (ρ).

**Method (synthetic simulation against sector returns, not pure LLM output):**
1. Download Fama-French 12 Industry Portfolios (value-weighted monthly returns) from Ken French's Data Library
2. Map each persona's career type to their closest Fama-French industry
3. For each persona, simulate 120 months of income shocks using their `income_volatility_sigma` (σ)
4. Run OLS regression: `ΔIncome_t = α + β × r_sector_t + ε_t`
5. Compute ρ = `Corr(income_shocks, r_sector)` from the same simulation

Using sector returns instead of the broad market gives a more economically meaningful β —
a tech executive's income is more correlated with tech sector movements than with the S&P 500.

**Interpreting β:**
- β ≈ 0 → income uncorrelated with sector (bond-like) — e.g. tenured professor, government worker
- β ≈ 0.5 → partially market-correlated (mixed) — e.g. bonus-driven finance professional
- β ≈ 1+ → income moves with or exceeds sector swings (equity-like) — e.g. tech exec with RSUs

**Interpreting ρ:**
- ρ is consumed by the Risk Agent to compute HC-correlation adjusted sector limits:
  `adjusted_limit = base_limit × (1 − ρ)`
- ⚠ Formula pending confirmation with Niha before Wednesday

**Data sources:**
- Sector returns: [Ken French Data Library — 12 Industry Portfolios](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html) (free)
- Income shocks: synthetic simulation seeded by σ per persona

In [13]:
import zipfile, io, re

# --- Step 1: Download Fama-French 12 Industry Portfolios ---
FF_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/12_Industry_Portfolios_CSV.zip"
response = requests.get(FF_URL, timeout=30)
zf = zipfile.ZipFile(io.BytesIO(response.content))

csv_name = [f for f in zf.namelist() if f.endswith('.CSV') or f.endswith('.csv')][0]
with zf.open(csv_name) as f:
    raw = f.read().decode('utf-8', errors='ignore').splitlines()

# Find value-weighted block
start = next(i for i, line in enumerate(raw) if 'Average Value Weighted Returns -- Monthly' in line)
data_start = start + 2

# Find end of block (next blank line after data starts)
end = next(i for i, line in enumerate(raw) if i > data_start and line.strip() == '')

# Parse using column names from line immediately after the block header
col_line = raw[start + 1].strip().lstrip(',')
cols = [c.strip() for c in col_line.split(',')]

rows = []
for line in raw[data_start:end]:
    parts = [p.strip() for p in line.split(',')]
    if len(parts) == len(cols) + 1 and re.match(r'^\d{6}$', parts[0]):
        rows.append([parts[0]] + parts[1:])

ff_df = pd.DataFrame(rows, columns=['date'] + cols)
ff_df['date'] = pd.to_datetime(ff_df['date'], format='%Y%m')
ff_df = ff_df.set_index('date')
ff_df = ff_df.apply(pd.to_numeric, errors='coerce') / 100  # % → decimal
ff_df = ff_df.replace(-99.99 / 100, np.nan).replace(-999 / 100, np.nan)
ff_df = ff_df.sort_index()

print(f"FF12 loaded: {ff_df.shape}")
print(f"Date range: {ff_df.index.min().date()} → {ff_df.index.max().date()}")
print(f"Industries: {list(ff_df.columns)}")

# --- Step 2: Career → FF12 industry mapping ---
CAREER_TO_FF_INDUSTRY = {
    "Academia":      "Hlth",
    "Technology":    "BusEq",
    "Finance":       "Money",
    "Healthcare":    "Hlth",
    "Medical":       "Hlth",
    "Legal":         "Other",
    "Real Estate":   "Other",
    "Military":      "Other",
    "Government":    "Other",
    "Arts":          "Other",
    "Engineering":   "BusEq",
    "Retail":        "Shops",
    "Energy":        "Enrgy",
    "Utilities":     "Utils",
    "Manufacturing": "Manuf",
    "Nonprofit":     "Other",
    "Hospitality":   "Shops",
}

def get_ff_industry(career_type):
    for key, ff in CAREER_TO_FF_INDUSTRY.items():
        if key.lower() in career_type.lower():
            return ff
    return None

# --- Step 3: Load CRSP broad market as fallback ---
crsp = pd.read_csv('crsp_market_index.csv', parse_dates=['date'])
crsp = crsp.set_index('date').sort_index()
crsp.index = crsp.index.to_period('M').to_timestamp()

# --- Step 4: Estimate β and ρ for each persona ---
def estimate_hc_beta(persona, ff_df, crsp, seed=None):
    sigma       = INCOME_VOLATILITY_SIGMA[persona['income_stability']]
    ff_industry = get_ff_industry(persona.get('career_type', ''))

    if ff_industry and ff_industry in ff_df.columns:
        sector_returns = ff_df[ff_industry].dropna().tail(120).values
        source = ff_industry
    else:
        sector_returns = crsp['vwretd'].dropna().tail(120).values
        source = "CRSP vwretd (fallback)"

    n = len(sector_returns)
    rng = np.random.default_rng(seed=seed)
    income_shocks = sigma * rng.standard_normal(n)

    slope, intercept, r_value, p_value, std_err = stats.linregress(sector_returns, income_shocks)

    # ρ — Corr(income_shocks, r_sector) from the same simulation
    # ⚠ pending confirmation with Niha that this is the intended computation
    correlation = round(float(np.corrcoef(income_shocks, sector_returns)[0, 1]), 4)

    return {
        "client_id":               persona["client_id"],
        "ff_industry":             source,
        "income_equity_beta":      round(slope, 4),       # renamed from human_capital_beta
        "income_equity_correlation": correlation,          # ρ — new field
        "beta_r_squared":          round(r_value ** 2, 4),
        "beta_p_value":            round(p_value, 4),
        "beta_std_err":            round(std_err, 4)
    }

beta_results = []
for i, persona in enumerate(all_raw_personas):
    result = estimate_hc_beta(persona, ff_df, crsp, seed=42 + i)
    beta_results.append(result)
    print(f"{result['client_id']:45s}  β={result['income_equity_beta']:+.4f}  "
          f"ρ={result['income_equity_correlation']:+.4f}  "
          f"R²={result['beta_r_squared']:.4f}  sector={result['ff_industry']}")

FF12 loaded: (1198, 12)
Date range: 1926-07-01 → 2026-04-01
Industries: ['NoDur', 'Durbl', 'Manuf', 'Enrgy', 'Chems', 'BusEq', 'Telcm', 'Utils', 'Shops', 'Hlth', 'Money', 'Other']
persona_biology_professor                      β=-0.0505  ρ=-0.0556  R²=0.0031  sector=Hlth
persona_tech_executive                         β=+0.6897  ρ=+0.0950  R²=0.0090  sector=BusEq
persona_financial_professional                 β=+0.0033  ρ=+0.0009  R²=0.0000  sector=Money
persona_emergency_room_physician               β=+0.0209  ρ=+0.0183  R²=0.0003  sector=Hlth
persona_corporate_litigation_attorney          β=+0.0140  ρ=+0.0132  R²=0.0002  sector=CRSP vwretd (fallback)
persona_commercial_real_estate_broker          β=-0.1056  ρ=-0.0146  R²=0.0002  sector=Other
persona_army_colonel                           β=-0.0372  ρ=-0.0404  R²=0.0016  sector=Other
persona_federal_policy_analyst                 β=+0.0797  ρ=+0.0885  R²=0.0078  sector=Other
persona_freelance_graphic_designer             β=+0.7512  ρ=+

## 10. Attach Beta & Correlation Estimates to Profiles & Run Sanity Check

Beta and correlation estimates from the OLS regression are attached back to each profile.
A sanity check confirms that the direction of `income_equity_beta` is consistent with the
expected human capital type:

- `bond-like` personas (σ = 0.05) → β expected near zero (low market sensitivity)
- `mixed` personas (σ = 0.20) → β expected moderate
- `equity-like` personas (σ = 0.40) → β expected highest in magnitude

Note: because income shocks are synthetic (not observed wages), β and ρ reflect
the structural relationship between earnings volatility and sector returns —
not an empirically measured income series. This is flagged here for the paper.

In [14]:
# Build beta and correlation lookup dicts
beta_lookup        = {r['client_id']: r['income_equity_beta']        for r in beta_results}
correlation_lookup = {r['client_id']: r['income_equity_correlation']  for r in beta_results}

# Rebuild all profiles with beta and correlation populated
all_profiles = []
for p in all_raw_personas:
    if p['income_stability'] not in INCOME_VOLATILITY_SIGMA:
        print(f"SKIP: {p['client_id']} — invalid income_stability value: {p['income_stability']}")
        continue
    beta        = beta_lookup.get(p['client_id'], None)
    correlation = correlation_lookup.get(p['client_id'], None)
    all_profiles.append(build_profile(p, beta=beta, correlation=correlation))

print(f"Rebuilt {len(all_profiles)} profiles with beta and correlation estimates\n")

# Beta sanity check — direction should be consistent with HC type
print("=== Beta Sanity Check ===")

for p in all_profiles:
    beta   = p['income_equity_beta']
    hc_type = p['human_capital_type']

    if hc_type == "bond-like" and abs(beta) > 0.3:
        flag = "FLAG — unexpectedly high beta for bond-like income"
    elif hc_type == "equity-like" and abs(beta) < 0.01:
        flag = "FLAG — unexpectedly low beta for equity-like income"
    else:
        flag = "OK"

    print(f"{flag:55s} {p['client_id']:45s}  β={beta:+.4f}  ρ={p['income_equity_correlation']:+.4f}  type={hc_type}")

Rebuilt 15 profiles with beta and correlation estimates

=== Beta Sanity Check ===
OK                                                      persona_biology_professor                      β=-0.0505  ρ=-0.0556  type=bond-like
OK                                                      persona_tech_executive                         β=+0.6897  ρ=+0.0950  type=equity-like
OK                                                      persona_financial_professional                 β=+0.0033  ρ=+0.0009  type=mixed
OK                                                      persona_emergency_room_physician               β=+0.0209  ρ=+0.0183  type=bond-like
OK                                                      persona_corporate_litigation_attorney          β=+0.0140  ρ=+0.0132  type=bond-like
OK                                                      persona_commercial_real_estate_broker          β=-0.1056  ρ=-0.0146  type=equity-like
OK                                                      persona_army_colonel 

## 11. Save Profiles & Beta Results to JSON

In [15]:
import os
os.makedirs('agents/profile', exist_ok=True)

# Save baseline and full set separately
baseline_profiles = all_profiles[:3]
with open('agents/profile/profiles_baseline.json', 'w') as f:
    json.dump(baseline_profiles, f, indent=2)

with open('agents/profile/profiles_all.json', 'w') as f:
    json.dump(all_profiles, f, indent=2)

# Save GPT-4o review log as audit trail
with open('agents/profile/persona_review_log.json', 'w') as f:
    json.dump(review_results, f, indent=2)

# Save beta and correlation estimation results separately for inspection
with open('agents/profile/beta_estimates.json', 'w') as f:
    json.dump(beta_results, f, indent=2)

print(f"Saved {len(baseline_profiles)} baseline profiles → agents/profile/profiles_baseline.json")
print(f"Saved {len(all_profiles)} total profiles       → agents/profile/profiles_all.json")
print(f"Saved GPT-4o review log                       → agents/profile/persona_review_log.json")
print(f"Saved beta & correlation estimates             → agents/profile/beta_estimates.json")

Saved 3 baseline profiles → agents/profile/profiles_baseline.json
Saved 15 total profiles       → agents/profile/profiles_all.json
Saved GPT-4o review log                       → agents/profile/persona_review_log.json
Saved beta & correlation estimates             → agents/profile/beta_estimates.json


## 12. Summary Table

In [16]:
summary = pd.DataFrame([{
    'Client': p['client_id'].replace('persona_', '').replace('_', ' ').title(),
    'Age': p['age'],
    'Financial Capital': f"${p['financial_capital']:,.0f}",
    'Human Capital': f"${p['human_capital_valuation']:,.0f}",
    'Total Wealth': f"${p['total_wealth']:,.0f}",
    'HC %': f"{p['human_capital_pct_of_total']}%",
    'HC Type': p['human_capital_type'],
    'σ': p['income_volatility_sigma'],
    'β': p['income_equity_beta'],
    'ρ': p['income_equity_correlation'],
    'Implicit Equity Exp.': p['implicit_equity_exposure'],
    'Risk Budget': p['effective_risk_budget'],
    'Equity Target': p['portfolio_equity_target'],
    'Risk Tolerance': p['risk_tolerance_level']
} for p in all_profiles])

print(summary.to_string(index=False))

                       Client  Age Financial Capital Human Capital Total Wealth  HC %     HC Type    σ       β       ρ  Implicit Equity Exp.  Risk Budget  Equity Target Risk Tolerance
            Biology Professor   45          $350,000    $1,236,825   $1,586,825 77.9%   bond-like 0.05 -0.0505 -0.0556                 0.039        0.961          0.922       Moderate
               Tech Executive   38        $2,000,000    $6,187,264   $8,187,264 75.6% equity-like 0.40  0.6897  0.0950                 0.302        0.698          0.396     Aggressive
       Financial Professional   35          $800,000    $4,077,017   $4,877,017 83.6%       mixed 0.20  0.0033  0.0009                 0.167        0.833          0.666     Aggressive
     Emergency Room Physician   42        $1,850,000    $4,956,508   $6,806,508 72.8%   bond-like 0.05  0.0209  0.0183                 0.036        0.964          0.928       Moderate
Corporate Litigation Attorney   38          $920,000    $4,408,425   $5,328,425 

## 13. Interpreting the Profile Results

### What the Numbers Mean

**Human Capital Valuation** is the present value of each client's future earnings, discounted
at the live 10-year Treasury yield pulled from FRED at runtime. A 28-year-old software engineer
earning \$320,000 with 37 years left has over \$3.3M in human capital — their single largest
asset, yet invisible on a traditional balance sheet.

**Income Volatility (σ)** measures earnings uncertainty:
- σ = 0.05: Near-certain income (tenured academics, government workers, military, nurses)
- σ = 0.20: Moderate uncertainty (bonus-driven but salaried: engineers, sales reps, electricians)
- σ = 0.40: High uncertainty (commission, RSU, or owner-operator income: tech exec, broker, designer)

**Income Equity Beta (β)** estimates how sensitive each persona's income stream is to
their sector's equity returns, computed via OLS regression of synthetic income shocks
against actual Fama-French 12 industry portfolio returns:
- β ≈ 0: income uncorrelated with sector movements (bond-like) — e.g. biology professor (−0.05)
- β moderate: partially market-correlated (mixed) — e.g. financial professional (+0.003)
- β high: income moves with sector swings (equity-like) — e.g. tech executive (+0.69)

**Income Equity Correlation (ρ)** is the Pearson correlation between income shocks and sector
returns from the same simulation. Consumed by the Risk Agent to compute HC-correlation adjusted
sector limits: `adjusted_limit = base_limit × (1 − ρ)`. A client whose income is highly
correlated with the technology sector faces a tighter tech allocation limit than one whose
income is uncorrelated.

Note: R² values are near zero by design — income shocks are synthetic random draws scaled
by σ, not observed wage series. β and ρ capture structural sensitivity, not empirical
income-market correlation. This limitation is flagged here for the paper and oral defense.

**Effective Risk Budget** answers: *how much additional equity risk can this client's portfolio absorb?*
It accounts for both the size and the type of human capital:
- The **Federal Policy Analyst** (σ = 0.05, risk budget = 0.954) has bond-like income — their
  portfolio can hold significant equity risk because their HC already acts as a large, stable
  fixed-income position
- The **Freelance Graphic Designer** (σ = 0.40, risk budget = 0.614) has volatile, equity-like
  income — adding more equity in the portfolio doubles down on the same risk factor
- The **Tech Executive** (σ = 0.40, β = +0.69) is already heavily exposed to tech through
  RSUs and income — the portfolio must hedge, not amplify, that concentration

**Portfolio Equity Target** is the starting equity weight handed to the Allocation Agent:
`effective_risk_budget − implicit_equity_exposure`. It represents the residual equity capacity
after subtracting the equity risk already embedded in the client's human capital. The Allocation
Agent builds the portfolio around this number — it is not a suggestion, it is a constraint.

## 14. What Gets Passed Downstream

Each profile passes downstream as a JSON object. Different agents consume different fields:

**Allocation Agent:**
- `human_capital_valuation` and `effective_risk_budget` → drives portfolio weight construction
- `portfolio_equity_target` → starting equity weight; residual risk capacity after subtracting implicit HC equity exposure
- `implicit_equity_exposure` → fraction of total wealth behaving like equity from income alone; caps the equity allocation before even looking at the portfolio
- `income_equity_beta` → adjusts equity allocation to avoid doubling sector risk in income + portfolio
- `current_holdings` → baseline the Allocation Agent rebalances from
- `investment_objective` → anchors the return target (Growth / Income / Preservation)

**Risk Agent:**
- `income_volatility_sigma` → parameterises earnings stress scenarios
- `income_equity_beta` → scales income shock in drawdown simulations
- `income_equity_correlation` → drives HC-correlation adjusted sector limits: `adjusted_limit = base_limit × (1 − ρ)`
- `implicit_equity_exposure` → sets the baseline equity risk already embedded in the client's human capital before portfolio construction
- `RSU_concentration` → flags single-stock concentration risk
- `liquidity_needs` → sets minimum cash/liquid asset floor

**Compliance Agent:**
- `human_capital_type` → classifies income stream as bond-like / mixed / equity-like, driving portfolio allocation constraints
- `industry_exposure_sector` → flags sector concentration against regulatory limits
- `risk_tolerance_level`, `investment_horizon_years`, `age` → suitability checks (FINRA Rule 2111)
- `RSU_concentration` → triggers concentration breach check if above threshold